In [1]:
# Load environment variables from .env (expects ANTHROPIC_API_KEY)
from dotenv import load_dotenv
from anthropic import Anthropic
from IPython.display import Markdown, display

load_dotenv()

True

In [2]:
client = Anthropic()

# Instantiate the client (reads ANTHROPIC_API_KEY from the environment)
# model = "claude-3-haiku-20240307"
model = "claude-haiku-4-5-20251001"

# Append a user turn to the conversation history
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

# Append an assistant turn to the conversation history
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Send the full conversation history and return the assistant reply
def chat(messages, system=None, temperature=0, stop_sequences=None):
    kwargs = dict(model=model,
                  max_tokens=1000,
                  messages=messages,
                  temperature=temperature)
    if system:
        kwargs["system"] = system
    if stop_sequences:
        kwargs["stop_sequences"] = stop_sequences if isinstance(stop_sequences, list) else [stop_sequences]
    message = client.messages.create(**kwargs)
    return message.content[0].text


In [3]:
messages = []

add_user_message(messages, "generate a very short event bridge rule as a JSON")

add_assistant_message(messages, "```json")

message = chat(messages, system=None, temperature=0, stop_sequences="```")

message

'\n{\n  "Name": "MyRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [4]:
import json

# Clean up and parse the JSON
clean_json = json.loads(message.strip())

In [5]:
clean_json

{'Name': 'MyRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

In [6]:
messages = []

prompt = "Output exactly three different short AWS CLI commands, one per line. No prose, no comments, no markdown fences, no numbering. Just the three commands separated by newlines."

add_user_message(messages, prompt)

text = chat(messages, temperature=0)
print(text)


aws s3 ls s3://my-bucket
aws ec2 describe-instances --region us-east-1
aws dynamodb list-tables --region us-west-2
